# 02 — Benchmark · Calibration · Conformal · XAI (synthetic demo)

> **Honesty banner — read first.** Every number in this notebook is produced on a **SYNTHETIC**
> dataset (60,000 rows, `scripts/make_synthetic.py`, pre/post-COVID Week-60 default shock).
> It exists to demonstrate the *mechanics* — model comparison, isotonic calibration, conformal
> triage, SHAP + DiCE, PSI drift — end to end. **None of these metrics describe the real Kaggle
> data.** The locked real-data numbers (W80–91, trials=5, 2026-11-09 user machine) are:
> ROC-AUC **0.8540** · PR-AUC 0.1468 · ECE 0.0165 · weekly-Gini stability **0.6843** (slope **+0.0044**/wk).
> See [`docs/REAL_RUN.md`](../docs/REAL_RUN.md). If anything here disagrees with that file, the file wins.

## 0. Setup — regenerate the synthetic data and build features (deterministic, ~2–4 min)

In [1]:
import sys
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

REPO = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "scripts"))
sys.path.insert(0, str(REPO / "src"))

from make_synthetic import generate          # noqa: E402
from prismrisk import features as F          # noqa: E402
from prismrisk.config import load_config     # noqa: E402

cfg = load_config()
FIG = REPO / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

WORK = Path(tempfile.mkdtemp(prefix="prismrisk_nb02_"))
raw = WORK / "raw"
raw.mkdir()
t = time.time()
tables = generate(rows=60_000, seed=42)
for name, frame in tables.items():
    frame.write_parquet(raw / name)
print(f"raw tables written in {time.time() - t:.1f}s")

t = time.time()
feat_path = F.build_features(raw, WORK / "features.parquet")
df = pl.read_parquet(feat_path)
print(f"features built in {time.time() - t:.1f}s -> {df.shape[0]} rows x {df.shape[1]} cols")
print("week span:", df["WEEK_NUM"].min(), "->", df["WEEK_NUM"].max(), "| default rate:", round(float(df["target"].mean()), 4))

raw tables written in 0.1s
features built in 0.1s -> 60000 rows x 34 cols
week span: 0 -> 91 | default rate: 0.0701


## 1. Four candidates, one lock — train on W0–59, select on W70–79, never touch W80–91 yet

Section 1 reports all four candidates. The **reference ("champion") model** used for the rest of the
notebook is the best **tree** candidate by val ROC-AUC — SHAP/DiCE need a tree model, and the
competition-style story is a tree champion anyway (the real-run champion was `xgboost_unconstrained`).

In [2]:
from lightgbm import LGBMClassifier                                    # noqa: E402

from prismrisk import metrics as M                                      # noqa: E402
from prismrisk.split import make_splits                                  # noqa: E402
from prismrisk.train import (                                           # noqa: E402
    TableEncoder, make_base_models, monotone_vector, split_metrics, xy,
)

splits = make_splits(df, week_col=cfg.week_col)
enc = TableEncoder().fit(splits["train"], cfg)
Xtr, ytr, wtr, amttr = xy(splits["train"], cfg, enc)
Xc, yc, wc, amtc = xy(splits["calib"], cfg, enc)
Xv, yv, wv, amtv = xy(splits["val"], cfg, enc)
Xt, yt, wt, amtt = xy(splits["test"], cfg, enc)
mcv = monotone_vector(enc.feature_names, cfg)
print(f"features={len(enc.feature_names)}  train={len(ytr)}  calib={len(yc)}  val={len(yv)}  test={len(yt)}")

candidates = make_base_models(cfg.random_seed)
candidates["lightgbm_monotone"] = LGBMClassifier(   # default params here (Optuna not run in-notebook)
    n_estimators=300, learning_rate=0.05, num_leaves=63,
    monotone_constraints=mcv, random_state=cfg.random_seed, n_jobs=-1, verbose=-1,
)

rows, test_gini = {}, {}
for name, model in candidates.items():
    t0 = time.time()
    model.fit(Xtr, ytr)
    fit_s = time.time() - t0
    p_val = model.predict_proba(Xv)[:, 1]
    t0 = time.time()
    p_lat = model.predict_proba(Xv[:2_000])[:, 1]
    ms_per_1k = (time.time() - t0) / len(p_lat) * 1_000
    p_test = model.predict_proba(Xt)[:, 1]
    rows[name] = {
        "val_roc_auc": M.roc_auc(yv, p_val),
        "val_pr_auc": M.pr_auc(yv, p_val),
        "test_gini_mean": M.weekly_gini_stability(wt, yt, p_test)["mean_gini"],
        "fit_s": round(fit_s, 1),
        "ms_per_1k_rows": round(ms_per_1k, 1),
    }

table = pd.DataFrame(rows).T.sort_values("val_roc_auc", ascending=False).round(4)
print(table.to_string())
tree_names = ["xgboost_unconstrained", "lightgbm_monotone"]
champion = table.loc[tree_names]["val_roc_auc"].idxmax()
print(f"\noverall best val ROC-AUC: {table.index[0]}")
print(f"reference champion for sections 2-6 (tree, for SHAP/DiCE): {champion}  (ties/within-0.0005 -> report both, never fudge)")

/home/user/PrismRisk/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


features=30  train=39136  calib=6520  val=6520  test=7824


                       val_roc_auc  val_pr_auc  test_gini_mean  fit_s  ms_per_1k_rows
logreg_baseline             0.8930      0.6811          0.8016    0.5             0.0
mlp_challenger              0.8799      0.6501          0.7683    2.7             0.0
lightgbm_monotone           0.8669      0.6303          0.7600    1.9             0.0
xgboost_unconstrained       0.8325      0.5769          0.7088    2.5             0.0

overall best val ROC-AUC: logreg_baseline
reference champion for sections 2-6 (tree, for SHAP/DiCE): lightgbm_monotone  (ties/within-0.0005 -> report both, never fudge)


## 2. Locked W80–91 — weekly Gini (stability, not just average)

In [3]:
champion_model = candidates[champion]
test_metrics = split_metrics(champion_model, Xt, yt, wt, amtt, cfg=cfg)
print({k: round(v, 4) for k, v in test_metrics.items()})

from sklearn.metrics import roc_auc_score  # noqa: E402

weekly = {}
for w in np.unique(wt):
    mask = wt == w
    if np.unique(yt[mask]).size < 2:
        continue
    weekly[int(w)] = 2.0 * roc_auc_score(yt[mask], champion_model.predict_proba(Xt[mask])[:, 1]) - 1.0

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(list(weekly), list(weekly.values()), color="#2563eb")
ax.axhline(0, color="#111", lw=0.8)
ax.set_xlabel("test week"); ax.set_ylabel("weekly Gini")
ax.set_title(f"Weekly Gini — {champion} on locked W80–91 (synthetic demo)")
fig.tight_layout()
fig.savefig(FIG / "weekly_gini.png", bbox_inches="tight")
plt.close(fig)
print("saved reports/figures/weekly_gini.png")

{'roc_auc': 0.8802, 'pr_auc': 0.6567, 'gini': 0.7604, 'gini_stability': 0.7423, 'gini_stability_custom': 0.7423, 'gini_week_min': 0.7082, 'brier': 0.0927, 'ece': 0.0908}


saved reports/figures/weekly_gini.png


## 3. Isotonic calibration — reliability before/after on locked weeks

In [4]:
from prismrisk.train import calibrate  # noqa: E402

iso, tau_star, cal_summary, test_cal = calibrate(
    champion_model, Xc=Xc, yc=yc, amt_c=amtc,
    X_val=Xv, y_val=yv, X_test=Xt, y_test=yt, w_test=wt, cfg=cfg,
)
print("calibration summary (val):", {k: round(v, 4) for k, v in cal_summary.items()})
print("locked W80-91 (calibrated):", {k: round(v, 4) for k, v in test_cal.items()})

p_raw = champion_model.predict_proba(Xt)[:, 1]
p_cal = iso.predict(p_raw)
bins = np.linspace(0, 1, 11)
rows_rel = []
for lo, hi in zip(bins[:-1], bins[1:], strict=True):
    mask = (p_raw >= lo) & (p_raw < hi)
    if mask.sum() == 0:
        continue
    rows_rel.append((p_raw[mask].mean(), yt[mask].mean(), "before"))
for lo, hi in zip(bins[:-1], bins[1:], strict=True):
    mask = (p_cal >= lo) & (p_cal < hi)
    if mask.sum() == 0:
        continue
    rows_rel.append((p_cal[mask].mean(), yt[mask].mean(), "after"))

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.plot([0, 1], [0, 1], "--", color="#999", lw=1, label="perfect")
for label, color in (("before", "#dc2626"), ("after", "#16a34a")):
    xs = [p for p, _, s in rows_rel if s == label]
    ys = [d for _, d, s in rows_rel if s == label]
    ax.plot(xs, ys, "o-", color=color, label=label)
ax.set_xlabel("mean predicted PD"); ax.set_ylabel("empirical default rate")
ax.set_title("Reliability curve — before/after isotonic (locked W80–91, synthetic)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "reliability.png", bbox_inches="tight")
plt.close(fig)
print("saved reports/figures/reliability.png")

calibration summary (val): {'brier_before': 0.0926, 'brier_after': 0.0698, 'ece_before': 0.0894, 'ece_after': 0.0074, 'tau_star': 0.13, 'max_profit_calib_usd': 21900758.7272}
locked W80-91 (calibrated): {'roc_auc': 0.8779, 'pr_auc': 0.6324, 'brier_cal': 0.0673, 'ece_cal': 0.0066, 'gini_mean_gini': 0.7557, 'gini_std_gini': 0.0345, 'gini_min_gini': 0.7061, 'gini_slope_per_week': 0.0007, 'gini_residual_std_gini': 0.0344, 'gini_stability': 0.7385, 'gini_custom_stability': 0.7385, 'gini_n_weeks': 12.0}


saved reports/figures/reliability.png


## 4. Conformal grey-zone triage — profit vs the binary cutoff (honest either way)

MAPIE `SplitConformalClassifier` (90% coverage), conformalized on calib weeks only, evaluated on the
locked test weeks. If the 3-way policy loses to the binary cutoff on this distribution, the notebook
says so — the README never hides it.

In [5]:
from mapie.classification import SplitConformalClassifier       # noqa: E402

from prismrisk.triage import (                                  # noqa: E402
    AUTO_APPROVE, AUTO_DENY, GREY_ZONE, BundleClassifier, _profit, decisions_from_sets,
)

base = BundleClassifier({"model": champion_model, "isotonic": iso})
mapie = SplitConformalClassifier(estimator=base, confidence_level=cfg.conformal_confidence, prefit=True)
mapie.conformalize(Xc, yc)                              # calib weeks ONLY

_, sets_test = mapie.predict_set(Xt)
decisions = decisions_from_sets(sets_test)
counts = pd.Series(decisions).value_counts()
print("decision counts:", counts.to_dict())

p_test_cal = iso.predict(champion_model.predict_proba(Xt)[:, 1])
binary_profit = M.expected_net_profit(yt, p_test_cal, amtt, tau_star,
                                      margin=cfg.interest_margin, lgd=cfg.lgd)
conformal_profit = _profit(yt, amounts=amtt, p=p_test_cal,
                           approve_mask=decisions == AUTO_APPROVE,
                           grey_mask=decisions == GREY_ZONE,
                           tau=tau_star, margin=cfg.interest_margin,
                           lgd=cfg.lgd, review_cost=cfg.manual_review_cost)
uplift = conformal_profit - binary_profit
print(f"binary cutoff (${tau_star:.2f}) profit:  ${binary_profit:,.0f}")
print(f"3-way conformal policy profit: ${conformal_profit:,.0f}")
print(f"grey-zone uplift: ${uplift:+,.0f}   {'(+ ve -> conformal pays)' if uplift > 0 else '(- ve -> binary cutoff wins on this distribution)'}")

fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.bar(["binary cutoff", "conformal 3-way"], [binary_profit / 1e6, conformal_profit / 1e6],
       color=["#6b7280", "#2563eb"])
ax.set_ylabel("net profit on locked W80–91 ($M)")
ax.set_title("Grey-zone vs binary cutoff (synthetic demo)")
fig.tight_layout()
fig.savefig(FIG / "conformal_profit.png", bbox_inches="tight")
plt.close(fig)
print("saved reports/figures/conformal_profit.png")

decision counts: {'AUTO_APPROVE': 7082, 'AUTO_DENY': 511, 'GREY_ZONE_REVIEW': 231}


binary cutoff ($0.13) profit:  $27,059,704
3-way conformal policy profit: $25,286,814
grey-zone uplift: $-1,772,890   (- ve -> binary cutoff wins on this distribution)
saved reports/figures/conformal_profit.png


## 5. One applicant, end to end — SHAP reasons → DiCE recourse → verified letter

Pick a locked-test applicant the model would **deny / grey-zone** (that is the only case where
recourse is demand-side useful). The letter is rendered by the deterministic template — numbers can
only come from the measured facts; `grounding_passed` is machine-checked.

In [6]:
import shap                                                             # noqa: E402

from prismrisk import explain_recourse as ER                            # noqa: E402
from prismrisk.notice_guard import EvidenceFacts, render_notice          # noqa: E402

cand = int(np.where((p_test_cal >= tau_star) & (yt == 1))[0][0]) if ((p_test_cal >= tau_star) & (yt == 1)).any() \
      else int(np.argsort(-p_test_cal)[0])
X_row = pd.DataFrame(Xt[cand:cand + 1], columns=enc.feature_names)
print(f"chosen applicant: locked-week row {cand} | week {int(wt[cand])} | actual default {int(yt[cand])} | calibrated PD {p_test_cal[cand]:.3f}")

explainer = shap.TreeExplainer(champion_model)
raw = explainer.shap_values(X_row)
arr = np.asarray(raw[-1]) if isinstance(raw, list) else np.asarray(raw)
if arr.ndim == 3:
    arr = arr[:, :, -1]
contrib = arr[0]
ev = np.atleast_1d(explainer.expected_value)
base = float(ev[-1]) if ev.size > 1 else float(ev[0])
exp = shap.Explanation(values=contrib, base_values=base,
                       data=X_row.iloc[0].to_numpy(), feature_names=enc.feature_names)
shap.plots.waterfall(exp, max_display=12, show=False)
plt.savefig(FIG / "shap_waterfall.png", bbox_inches="tight")
plt.close()
print("saved reports/figures/shap_waterfall.png")

Xtr_df = pd.DataFrame(Xtr, columns=enc.feature_names)
rec = ER.recourse(champion_model, Xtr_df, X_row, total_cfs=2, bg_rows=300, seed=42, calibrator=iso)
print("feasible:", rec["feasible"], "| base_wrapper_pd:", rec.get("base_wrapper_pd"), "| features_varied:", rec.get("features_varied"))
print("imputed for DiCE:", rec.get("imputed_features"))
for cf in rec["counterfactuals"]:
    print("  CF: relative_change=%.2f  changes=%s" % (cf["relative_change"], list(cf["changes"])))

chosen applicant: locked-week row 21 | week 89 | actual default 1 | calibrated PD 0.640


/home/user/PrismRisk/.venv/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


saved reports/figures/shap_waterfall.png


/home/user/PrismRisk/.venv/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


/home/user/PrismRisk/.venv/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  6.04it/s]

100%|██████████| 1/1 [00:00<00:00,  6.01it/s]

feasible: True | base_wrapper_pd: 0.6397 | features_varied: ['static_past_dpd_001P', 'mainoccupationinc_384A', 'annuity_780A', 'birth_259_days', 'maininc_clean', 'prev_max_appl_annuity_001A', 'max_bureau_origamount_003A', 'credamount_770A']
imputed for DiCE: ['risk_band_pct']
  CF: relative_change=2.06  changes=['static_past_dpd_001P', 'mainoccupationinc_384A', 'annuity_780A', 'birth_259_days', 'maininc_clean', 'max_bureau_origamount_003A', 'credamount_770A']
  CF: relative_change=2.56  changes=['static_past_dpd_001P', 'mainoccupationinc_384A', 'annuity_780A', 'birth_259_days', 'maininc_clean', 'max_bureau_origamount_003A', 'credamount_770A']


In [7]:
reasons = rec["top_reasons"]
amt = float(X_row["credamount_770A"].iloc[0])
if rec["counterfactuals"]:
    cf0 = rec["counterfactuals"][0]
    p_new = float(cf0["wrapper_pd"])   # calibrated PD from the SAME wrapper + imputed basis DiCE used
    target_amt = float(cf0["changes"].get("credamount_770A", {}).get("to", amt))
else:
    cf0, p_new, target_amt = None, p_test_cal[cand], amt
print(f"wrapper-scored PD after CF: {p_new:.1%}  (applicant before: {p_test_cal[cand]:.1%})")

if cf0 is not None and target_amt < amt and p_new < p_test_cal[cand]:
    # deterministic_letter's wording assumes an amount-reduction CF — template fits here
    facts = EvidenceFacts(
        pd_pct=f"{p_test_cal[cand] * 100:.1f}%",
        tau_pct=f"{tau_star * 100:.1f}%",
        current_loan=f"${amt:,.0f}",
        target_loan=f"${target_amt:,.0f}",
        new_pd_pct=f"{p_new * 100:.1f}%",
        reason_1=reasons[0]["label"],
        reason_2=reasons[1]["label"] if len(reasons) > 1 else "overall application strength",
        reason_3=reasons[2]["label"] if len(reasons) > 2 else "overall application strength",
        decision=str(decisions[cand]),
        applicant_ref=f"applicant W{int(wt[cand])}",
    )
    notice = render_notice(facts, llm=None)
    print("grounding_passed:", notice.grounding_passed, "| fallback_triggered:", notice.fallback_triggered)
    print()
    print(notice.text)
else:
    # honest path: template would lie ("reducing the amount" when it increases) — print raw verified facts
    print("NOTE: the feasible CF does NOT reduce the requested amount (synthetic artifact: amount ~ income),")
    print("so the deterministic adverse-action template would misstate it. Raw measured facts instead:")
    print(f"  applicant: week {int(wt[cand])} | calibrated PD {p_test_cal[cand]:.1%} vs tau {tau_star:.1%} | decision {decisions[cand]}")
    print(f"  wrapper-scored PD after CF: {p_new:.1%}")
    for col, delta in cf0["changes"].items():
        print(f"    {col}: {delta['from']:.4g} -> {delta['to']:.4g}")
    print("  reasons:", "; ".join(r["label"] for r in reasons))

wrapper-scored PD after CF: 1.2%  (applicant before: 64.0%)
grounding_passed: True | fallback_triggered: False

Dear applicant W89,

Thank you for your application. Our review model estimates your probability of default at 64.0%, above the 13.0% approval threshold. The main factors were: Static past dpd; Mainoccupationinc 384a; Prev max appl annuity 001a.

One action associated with a lower model-estimated risk is reducing the requested amount from $21,515 to $12,565 (estimated new PD 1.2%). This is a model decision, not a final legal determination; you may ask for manual review or provide additional documentation.

PrismRisk Credit Review


## 6. Weekly PSI — bins frozen on train weeks (the M5 monitor, streaming weekly)

In [8]:
from prismrisk import drift as D  # noqa: E402

Xall = enc.transform(df)
p_all = iso.predict(champion_model.predict_proba(Xall)[:, 1])
df_drift = df.with_columns(pl.Series("predicted_pd", p_all))

top5 = D.select_top_features(champion_model, Xall[::50], enc.feature_names, k=5)
top5 = [c for c in top5 if pl.Series(c, df_drift[c]).dtype.is_numeric()][:5]
print("top-5 monitored features (mean |SHAP|):", top5)

report = D.weekly_psi(df_drift, feature_cols=top5, pd_col="predicted_pd",
                      week_col="WEEK_NUM", train_hi=cfg.train_weeks[1], n_bins=10)
counts = report["band"].to_list()
bands = {b: counts.count(b) for b in sorted(set(counts))}
print("band counts across post-train weeks:", bands)

pd_series = report.filter(pl.col("feature") == "predicted_pd").sort("week")
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(pd_series["week"], pd_series["psi"], "o-", color="#2563eb")
ax.axhline(0.25, color="#ef4444", ls="--", lw=1.2, label="retrain threshold (0.25)")
ax.axhline(0.10, color="#f59e0b", ls="--", lw=1.0, label="warning threshold (0.10)")
ax.set_xlabel("week"); ax.set_ylabel("PSI (predicted PD)")
ax.set_title("Weekly PSI — predicted PD, bins frozen on train weeks (synthetic demo)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "psi_weekly.png", bbox_inches="tight")
plt.close(fig)
print("saved reports/figures/psi_weekly.png")

/home/user/PrismRisk/.venv/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


top-5 monitored features (mean |SHAP|): ['static_past_dpd_001P', 'mainoccupationinc_384A', 'birth_259_days', 'max_dpd_total_001P', 'total_overdue_amt']


band counts across post-train weeks: {'green': 192}
saved reports/figures/psi_weekly.png


## 7. Reading these honestly

1. **Every figure above is synthetic-demo.** The real pipeline, on the real locked W80–91, produced
   ROC-AUC **0.8540** / stability **0.6843** / slope **+0.0044** (docs/REAL_RUN.md) — notebook numbers
   will never match that and must never be quoted as if they did.
2. Optuna was **not** run inside this notebook (default LightGBM monotone params). The real champion
   came from a 5-trial Optuna search on the user machine; `train.py` is the production path.
3. The MLP candidate is in for completeness only — in the real run it was clearly inferior and
   should be considered for the chopping block (see IMPLEMENTATION_PLAN §13).
4. Recourse/CFs on synthetic data find flips easily; on real data they can be infeasible by design
   (`feasible=False`) — that is a feature, not a bug.
5. Artifacts used here: this notebook (executed), `reports/figures/*.png` (committed), no real data.